# Testing Shafer's Dichromatic Reflection Model for Specular Highlight Detection

This notebook lets you **randomly sample images from any dataset folder** and test the
classical (zero-training) specular-highlight detector based on **Shafer's dichromatic
reflection model (1985)** / Klinker et al.: a specular highlight is a pixel that is
**bright** (high Value) **and desaturated** (low Saturation), because the specular
component adds the (near-white) illuminant color on top of the diffuse (colored)
component, diluting saturation.

**Known result from prior testing** (see `results/MIPNERF_BOTTLENECK_PLAN_2026-07-01.md`
§6): this detector correctly flags TRUE specular highlights on real scenes (metal glints,
reflective lids, glossy sheens) but can ALSO false-positive on bright, desaturated **diffuse**
surfaces (white backgrounds, white plastic, light countertops) — because those satisfy the
same low-level statistics as a true highlight. Use this notebook to see exactly where each
effect shows up on the dataset/scene you care about.

**How to use:** set `URL_PATH` in the Config cell to any folder of images, e.g.:
```
dataset\mipnerf360\counter\images
dataset\Anisotropic-Synthetic-Dataset\plane\train
```
Every run re-samples **10 random images** from that folder (no fixed seed by default —
set `RANDOM_SEED` if you want a reproducible sample).

In [ ]:
import os
import sys
import glob
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Try to import the canonical implementation (single source of truth) from the repo's
# spec-fastgs/tools/ folder; fall back to an inline copy if the notebook is moved/run
# from elsewhere so it never hard-fails.
REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "spec-fastgs"))
try:
    from tools.classical_specular_mask import classical_specular_mask as _canonical_mask
    USING_CANONICAL_IMPL = True
except Exception as e:
    USING_CANONICAL_IMPL = False
    print(f"[info] could not import canonical tools/classical_specular_mask.py ({e});"
          f" using an inline copy of the same logic instead.")

    def _canonical_mask(img_path, sat_thresh=0.25, val_thresh=0.75):
        """Inline fallback identical to spec-fastgs/tools/classical_specular_mask.py."""
        img = np.array(Image.open(img_path).convert("RGB")).astype(np.float32) / 255.0
        maxc = img.max(axis=-1)
        minc = img.min(axis=-1)
        V = maxc
        S = np.where(maxc > 1e-6, (maxc - minc) / (maxc + 1e-6), 0.0)
        mask = (V > val_thresh) & (S < sat_thresh)
        overlay = (img * 255).astype(np.uint8).copy()
        overlay[mask] = [255, 0, 0]
        return mask, float(mask.mean()), overlay

print(f"repo root       : {REPO_ROOT}")
print(f"canonical impl? : {USING_CANONICAL_IMPL}")

## Config — EDIT THIS CELL

`URL_PATH` accepts either a relative path (from the repo root, `\` or `/` both work) or an
absolute path.

In [ ]:
# ============================================================
# EDIT ME
# ============================================================
URL_PATH = r"dataset\mipnerf360\counter\images"
# Other examples you can swap in:
# URL_PATH = r"dataset\Anisotropic-Synthetic-Dataset\plane\train"
# URL_PATH = r"dataset\Anisotropic-Synthetic-Dataset\teapot\train"
# URL_PATH = r"dataset\mipnerf360\garden\images"

NUM_SAMPLES  = 10      # how many random images to test each run
RANDOM_SEED  = None    # set an int (e.g. 42) for a reproducible sample; None = truly random each run

SAT_THRESH   = 0.25    # Shafer/HSV threshold: flag if Saturation < SAT_THRESH
VAL_THRESH   = 0.75    # Shafer/HSV threshold: flag if Value (brightness) > VAL_THRESH

MAX_DISPLAY_SIZE = 500  # longest side (px) for computation + display; keeps this fast on
                        # full-resolution real photos (Mip-NeRF images can be 4000px+)
# ============================================================

IMAGE_EXTS = ("*.png", "*.PNG", "*.jpg", "*.JPG", "*.jpeg", "*.JPEG")

## 1. List images in `URL_PATH` and draw a random sample

In [ ]:
folder = Path(URL_PATH)
if not folder.is_absolute():
    folder = REPO_ROOT / folder

if not folder.is_dir():
    raise FileNotFoundError(
        f"URL_PATH does not exist or is not a directory: {folder}\n"
        f"Check the path (relative to repo root {REPO_ROOT}) and try again."
    )

all_images = []
for pat in IMAGE_EXTS:
    all_images.extend(folder.glob(pat))
all_images = sorted(set(all_images))

if not all_images:
    raise FileNotFoundError(f"No images (png/jpg/jpeg) found directly inside: {folder}")

rng = random.Random(RANDOM_SEED)
n = min(NUM_SAMPLES, len(all_images))
if n < NUM_SAMPLES:
    print(f"[warn] only {len(all_images)} images available in this folder; "
          f"using all {n} instead of the requested {NUM_SAMPLES}.")
sampled = rng.sample(all_images, n)

print(f"folder          : {folder}")
print(f"total images    : {len(all_images)}")
print(f"sampled ({n}):")
for p in sampled:
    print("  ", p.name)

## 2. Run the Shafer dichromatic detector on each sampled image

In [ ]:
def load_resized(path, max_size):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    m = max(w, h)
    if m > max_size:
        s = max_size / m
        img = img.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BILINEAR)
    return img

def hsv_stats(img_rgb01, mask):
    """Mean V/S over the whole image vs. over just the flagged pixels (for context)."""
    maxc = img_rgb01.max(axis=-1); minc = img_rgb01.min(axis=-1)
    V = maxc; S = np.where(maxc > 1e-6, (maxc - minc) / (maxc + 1e-6), 0.0)
    out = {"mean_V_all": float(V.mean()), "mean_S_all": float(S.mean())}
    if mask.any():
        out["mean_V_flagged"] = float(V[mask].mean())
        out["mean_S_flagged"] = float(S[mask].mean())
    else:
        out["mean_V_flagged"] = float("nan")
        out["mean_S_flagged"] = float("nan")
    return out

results = []
for path in sampled:
    small = load_resized(path, MAX_DISPLAY_SIZE)
    tmp_path = path  # canonical fn re-reads from disk at native res; we instead run it on
                     # the resized array directly to keep this fast on big real photos.
    arr01 = np.array(small).astype(np.float32) / 255.0
    maxc = arr01.max(axis=-1); minc = arr01.min(axis=-1)
    V = maxc
    S = np.where(maxc > 1e-6, (maxc - minc) / (maxc + 1e-6), 0.0)
    mask = (V > VAL_THRESH) & (S < SAT_THRESH)
    overlay = (arr01 * 255).astype(np.uint8).copy()
    overlay[mask] = [255, 0, 0]
    stats = hsv_stats(arr01, mask)
    results.append({
        "name": path.name,
        "path": str(path),
        "image": np.array(small),
        "saturation_map": S,
        "mask": mask,
        "overlay": overlay,
        "flagged_pct": 100.0 * mask.mean(),
        **stats,
    })

print(f"Ran the detector on {len(results)} images (thresh: S<{SAT_THRESH}, V>{VAL_THRESH}).")

## 3. Visualize: original | saturation map | specular-candidate overlay

Look for two things in the overlay column:
- **Good signal**: red pixels tightly tracing glints on metal / glass / glossy plastic.
- **False positives**: large solid-red blobs over plain white/bright backgrounds or matte
  light-colored diffuse surfaces — this is the known confound (see notebook intro).

In [ ]:
n = len(results)
fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
if n == 1:
    axes = axes[None, :]

for i, r in enumerate(results):
    axes[i, 0].imshow(r["image"])
    axes[i, 0].set_title(f"{r['name']}\noriginal", fontsize=9)
    axes[i, 1].imshow(r["saturation_map"], cmap="viridis", vmin=0, vmax=1)
    axes[i, 1].set_title("saturation map\n(dark = desaturated)", fontsize=9)
    axes[i, 2].imshow(r["overlay"])
    axes[i, 2].set_title(f"specular candidates (red)\n{r['flagged_pct']:.2f}% of pixels", fontsize=9)
    for j in range(3):
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()

## 4. Summary statistics across the sample

In [ ]:
df = pd.DataFrame([
    {
        "image": r["name"],
        "flagged_%": round(r["flagged_pct"], 3),
        "mean_V_all": round(r["mean_V_all"], 3),
        "mean_S_all": round(r["mean_S_all"], 3),
        "mean_V_flagged": round(r["mean_V_flagged"], 3) if r["flagged_pct"] > 0 else float("nan"),
        "mean_S_flagged": round(r["mean_S_flagged"], 3) if r["flagged_pct"] > 0 else float("nan"),
    }
    for r in results
])
display(df)
print(f"\nfolder tested        : {folder}")
print(f"mean flagged % (n={n}) : {df['flagged_%'].mean():.3f}%")
print(f"min / max flagged %   : {df['flagged_%'].min():.3f}% / {df['flagged_%'].max():.3f}%")
print("\nInterpretation cue: a healthy 'true highlight' scene usually flags a SMALL, "
      "localized fraction of pixels (roughly 1-5%). A flagged_% that jumps to 30-80% on a "
      "single image is a strong sign of the bright-diffuse false-positive confound (e.g. a "
      "plain white background) rather than genuine specularity - open that image above to check.")

In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(df["image"], df["flagged_%"], color="crimson")
plt.ylabel("% of pixels flagged as specular-candidate")
plt.xticks(rotation=60, ha="right")
plt.title(f"Shafer dichromatic detector - flagged % per image\n{folder}")
plt.tight_layout()
plt.show()

## 5. (Optional) Threshold sensitivity sweep

Shows how sensitive the flagged fraction is to `SAT_THRESH`/`VAL_THRESH` on one representative
image from the sample — useful for judging how much of a knife-edge the classical threshold is
on this particular dataset/material mix.

In [ ]:
probe = results[0]  # change index to inspect a different sampled image
arr01 = probe["image"].astype(np.float32) / 255.0
maxc = arr01.max(axis=-1); minc = arr01.min(axis=-1)
V = maxc; S = np.where(maxc > 1e-6, (maxc - minc) / (maxc + 1e-6), 0.0)

sat_grid = np.linspace(0.05, 0.45, 9)
val_grid = np.linspace(0.55, 0.95, 9)
heat = np.zeros((len(val_grid), len(sat_grid)))
for vi, vt in enumerate(val_grid):
    for si, st in enumerate(sat_grid):
        heat[vi, si] = 100.0 * ((V > vt) & (S < st)).mean()

plt.figure(figsize=(6, 5))
im = plt.imshow(heat, origin="lower", aspect="auto", cmap="magma",
                extent=[sat_grid[0], sat_grid[-1], val_grid[0], val_grid[-1]])
plt.colorbar(im, label="% pixels flagged")
plt.scatter([SAT_THRESH], [VAL_THRESH], color="cyan", marker="x", s=120,
            label="current SAT_THRESH/VAL_THRESH")
plt.xlabel("SAT_THRESH (flag if S < this)")
plt.ylabel("VAL_THRESH (flag if V > this)")
plt.title(f"Threshold sensitivity on: {probe['name']}")
plt.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()